# NLP Basics & NLP Models — Assignments 1–3

We'll use a small hand-made dataset of 25 movie reviews throughout all three assignments.

**Contents**
1. Text Cleaning & Tokenization
2. TF-IDF Implementation
3. Word2Vec Embeddings


In [ ]:
# All the libraries we'll need across every assignment
import re
import numpy as np
import pandas as pd

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec


### Our dataset: 25 short movie reviews

In [ ]:
reviews = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "Terrible plot and bad acting, I would not recommend this film.",
    "The cinematography was stunning, but the story felt a bit slow.",
    "One of the best movies I have seen this year, truly amazing.",
    "I fell asleep halfway through, it was so boring.",
    "Great performances by the entire cast, very impressive direction.",
    "Waste of time and money, the plot made no sense at all.",
    "A heartwarming story with a wonderful message about family.",
    "The action scenes were thrilling and kept me on the edge of my seat.",
    "Poor script writing ruined what could have been a good movie.",
    "Amazing visuals and a gripping storyline, I was hooked from start to end.",
    "The dialogue felt forced and unnatural throughout the whole film.",
    "A delightful comedy that made me laugh out loud several times.",
    "The pacing was inconsistent and the ending felt rushed.",
    "Brilliant soundtrack and beautiful set design, a visual treat.",
    "I did not enjoy this movie, the characters were flat and boring.",
    "An emotional rollercoaster with a surprising and satisfying twist.",
    "The special effects were impressive but the story was weak.",
    "A must watch film with great acting and a powerful message.",
    "This was one of the worst movies I have ever watched.",
    "The romance between the leads felt genuine and touching.",
    "Too long and repetitive, could have been cut down significantly.",
    "A masterpiece of modern cinema, beautifully directed and acted.",
    "Disappointing sequel that fails to live up to the original.",
    "Wonderful family movie with great humor and heart."
]

print("Number of reviews:", len(reviews))
reviews[:3]


---
# Assignment 1: Text Cleaning & Tokenization

**Mental model:** Raw text is messy (mixed case, punctuation, filler words). We clean it step by step so a computer can work with the meaningful words only.

Steps: Lowercasing → Removing punctuation → Removing stopwords → Tokenization


### Step 1: Lowercasing

Why: "Movie" and "movie" are the same word to a human, but a computer treats them as different strings unless we lowercase everything first. This keeps our vocabulary consistent.


In [ ]:
def lowercase_text(text):
    return text.lower()


lowercased_reviews = [lowercase_text(review) for review in reviews]
print(lowercased_reviews[0])


### Step 2: Removing punctuation

Why: Punctuation marks like `.` `,` `!` don't usually carry meaning for basic text analysis, and they can cause the same word to be counted differently (e.g. "movie," vs "movie").


In [ ]:
def remove_punctuation(text):
    # Keep only letters, numbers, and spaces
    return re.sub(r'[^a-z0-9\s]', '', text)


no_punctuation_reviews = [remove_punctuation(review) for review in lowercased_reviews]
print(no_punctuation_reviews[0])


### Step 3: Tokenization

Why: Tokenization splits a sentence into individual words (tokens). This turns one long string into a list we can analyze word by word.


In [ ]:
def tokenize_text(text):
    return word_tokenize(text)


tokenized_reviews = [tokenize_text(review) for review in no_punctuation_reviews]
print(tokenized_reviews[0])


### Step 4: Removing stopwords

Why: Words like "the", "is", "a", "and" appear extremely often but carry very little meaning on their own. Removing them helps us focus on the words that actually distinguish one review from another.


In [ ]:
stop_words = set(stopwords.words('english'))
print("Example stopwords:", list(stop_words)[:10])
print("Total stopwords:", len(stop_words))


In [ ]:
def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]


cleaned_tokens = [remove_stopwords(tokens) for tokens in tokenized_reviews]
print("Before removing stopwords:", tokenized_reviews[0])
print("After removing stopwords: ", cleaned_tokens[0])


### Final cleaned tokens for all reviews

In [ ]:
for i, tokens in enumerate(cleaned_tokens[:5]):
    print(f"Review {i+1}: {tokens}")


### Explanation of Each Preprocessing Step

- **Lowercasing**: makes sure "Great" and "great" are treated as the exact same word, so word counts aren't split across different capitalizations.
- **Removing punctuation**: strips out symbols like `.` `,` `!` that don't add meaning for basic word-level analysis and would otherwise create noisy, duplicate-looking tokens (e.g. "movie" vs "movie!").
- **Tokenization**: breaks a full sentence string into a list of individual words, which is the format most NLP tools (TF-IDF, Word2Vec) expect as input.
- **Removing stopwords**: filters out very common but low-meaning words (like "the", "is", "a") so the remaining words better represent what each review is actually about.


---
# Assignment 2: TF-IDF Implementation

**Mental model:** TF-IDF gives every word in every document a score. A word gets a **high score** if it appears often in one document but rarely across all documents — meaning it's likely important/distinctive for that document.

Steps: Use `TfidfVectorizer` → Convert text into TF-IDF vectors → Display feature names, matrix shape, top words per document


### Step 1: Use sklearn's TfidfVectorizer on our dataset

In [ ]:
# TfidfVectorizer can do cleaning internally too, but we pass in our own cleaned text
# so the results reflect the exact preprocessing we did in Assignment 1
cleaned_reviews_text = [" ".join(tokens) for tokens in cleaned_tokens]

tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform(cleaned_reviews_text)

print("First cleaned review as text:", cleaned_reviews_text[0])


### Step 2: Display feature names

In [ ]:
feature_names = tfidf_vectorizer.get_feature_names_out()
print("Number of unique words (features):", len(feature_names))
print("First 20 feature names:", feature_names[:20])


### Step 3: Display the shape of the matrix

In [ ]:
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("(This means:", tfidf_matrix.shape[0], "documents x", tfidf_matrix.shape[1], "unique words)")


### Step 4: Top 10 important words per document

In [ ]:
def get_top_words_for_document(doc_index, top_n=10):
    # Get the TF-IDF scores for this one document as a plain array
    doc_scores = tfidf_matrix[doc_index].toarray().flatten()

    # Get the indexes of the highest scores, sorted from highest to lowest
    top_indexes = doc_scores.argsort()[::-1][:top_n]

    # Only keep words that actually have a non-zero score
    top_words = [(feature_names[i], round(doc_scores[i], 3)) for i in top_indexes if doc_scores[i] > 0]
    return top_words


for i in range(3):
    print(f"Review {i+1}: \"{reviews[i]}\"")
    print("Top words:", get_top_words_for_document(i))
    print()


### What does a high TF-IDF value mean?

A high TF-IDF score means a word appears **often in this specific document** but **rarely across the other documents** in the dataset. This makes it a good indicator of what makes that document distinctive — for example, the word "boring" scoring high in a negative review tells us that word is central to that review's meaning, and it doesn't show up much elsewhere.

### Why is IDF important?

TF (Term Frequency) alone would give high scores to words that are simply repeated a lot in one document — but some words (like "movie" or "film" in our dataset) appear in almost every review and aren't actually distinctive. IDF (Inverse Document Frequency) reduces the score of words that appear in many documents, so only genuinely distinguishing words end up with a high combined TF-IDF score. Without IDF, common-but-uninformative words could dominate the rankings.


---
# Assignment 3: Word2Vec Embeddings

**Mental model:** Instead of counting words (like TF-IDF), Word2Vec learns a dense numeric vector for each word based on the *other words that tend to appear around it*. Words used in similar contexts end up with similar vectors.


### Step 1: Train a simple Word2Vec model with gensim

In [ ]:
# Word2Vec expects a list of tokenized sentences - we already have this from Assignment 1
word2vec_model = Word2Vec(
    sentences=cleaned_tokens,
    vector_size=50,   # size of each word's vector
    window=3,         # how many neighboring words to consider as "context"
    min_count=1,      # include even words that appear only once (small dataset)
    workers=1,
    seed=42
)

print("Vocabulary size:", len(word2vec_model.wv.index_to_key))
print("Some words in vocabulary:", word2vec_model.wv.index_to_key[:15])


### Step 2: Find words similar to a chosen word ('good')

In [ ]:
# NOTE: with a tiny dataset like ours, these similarities are just a demo of how the method works,
# not necessarily meaningful — Word2Vec needs a lot more data to learn truly good relationships.
similar_words = word2vec_model.wv.most_similar("good", topn=5)
print("Words most similar to 'good':")
for word, score in similar_words:
    print(f"  {word}: {score:.3f}")


### Step 3: Vector size and shape for a word

In [ ]:
good_vector = word2vec_model.wv["good"]
print("Vector for 'good':")
print(good_vector)
print("\nVector shape:", good_vector.shape)
print("Vector size (dimensions):", word2vec_model.wv.vector_size)


### TF-IDF vs Word2Vec — Key Differences

- **What it represents**: TF-IDF represents each word as a single importance *score* per document; Word2Vec represents each word as a dense *vector* (list of numbers) capturing its meaning/context, independent of any single document.
- **Captures meaning?**: TF-IDF has no concept of word meaning or similarity — it only counts occurrences. Word2Vec captures semantic relationships, so words used in similar contexts (like "good" and "great") end up with similar vectors.
- **Vector size**: TF-IDF vectors are as long as the entire vocabulary (one number per unique word, mostly zeros — "sparse"). Word2Vec vectors have a small fixed size (e.g. 50 or 100 numbers) regardless of vocabulary size — "dense".
- **Depends on document context?**: TF-IDF scores are tied to a specific document (how important is this word *in this document*). Word2Vec vectors describe a word's general meaning learned from all the text it was trained on, not tied to one document.
- **Data needed**: TF-IDF works reasonably well even on small datasets. Word2Vec typically needs a much larger amount of text to learn meaningful, reliable word relationships (our tiny 25-review dataset is really just for practicing the mechanics).
- **Common use case**: TF-IDF is often used for search/ranking and simple document similarity. Word2Vec is used as a building block for deeper NLP tasks (sentiment analysis, recommendation systems, feeding into neural networks) where capturing word meaning matters.


### Bonus: Using a Pre-trained Model (Google News Vectors)

Loading Google's pre-trained `word2vec-google-news-300` model (trained on ~100 billion words) would give far more meaningful similarities than our tiny 25-review dataset can produce. The model file is large (~1.6GB) and needs to be downloaded separately — for example via `gensim.downloader`:

```python
import gensim.downloader as api
pretrained_model = api.load("word2vec-google-news-300")
print(pretrained_model.most_similar("good", topn=5))
```

This step is left as an optional bonus since it requires a large download and isn't needed to complete the core assignment.
